# 🧠 Notebook 2 — Data Scientist
## Forestry Image Classification | ISB46703

**Role:** Data Scientist  
**Tasks:**
- Create CNN models: ResNet50, DenseNet121, MobileNetV3
- Apply Transfer Learning + Hyperparameter Tuning
- Train each model for 50 epochs
- Record accuracy, mAP, and training time


## 📦 1. Install & Import Libraries

In [ ]:
!pip install tensorflow keras scikit-learn numpy matplotlib pillow tqdm

In [ ]:
import os
import time
import json
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import ResNet50, DenseNet121, MobileNetV3Small
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import average_precision_score, confusion_matrix, classification_report

print(f'✅ TensorFlow version : {tf.__version__}')
print(f'✅ GPU available      : {len(tf.config.list_physical_devices("GPU")) > 0}')

## ⚙️ 2. Configuration

In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────
BASE_DIR    = '../dataset'
RESULTS_DIR = '../results'
os.makedirs(RESULTS_DIR, exist_ok=True)

CLASSES     = ['healthy_forest', 'deforested', 'forest_fire', 'flooded_forest', 'diseased_forest']
NUM_CLASSES = len(CLASSES)
IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
EPOCHS      = 50
SEED        = 42

# Phase 1 — Feature extraction (frozen base)
LR_PHASE1   = 1e-3
# Phase 2 — Fine-tuning (unfreeze top layers)
LR_PHASE2   = 1e-5

print(f'📂 Dataset    : {BASE_DIR}')
print(f'🏷️  Classes    : {CLASSES}')
print(f'🖼️  Image size : {IMG_SIZE}')
print(f'📦 Batch size : {BATCH_SIZE}')
print(f'🔄 Epochs     : {EPOCHS}')

## 🔄 3. Data Generators (Augmentation)

In [ ]:
# ── DATA AUGMENTATION for Training ─────────────────────────────
train_datagen = ImageDataGenerator(
    rescale            = 1./255,
    rotation_range     = 20,
    width_shift_range  = 0.2,
    height_shift_range = 0.2,
    shear_range        = 0.15,
    zoom_range         = 0.2,
    horizontal_flip    = True,
    vertical_flip      = False,
    brightness_range   = [0.8, 1.2],
    fill_mode          = 'nearest'
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

# ── CREATE GENERATORS ──────────────────────────────────────────
train_gen = train_datagen.flow_from_directory(
    os.path.join(BASE_DIR, 'train'),
    target_size  = IMG_SIZE,
    batch_size   = BATCH_SIZE,
    class_mode   = 'categorical',
    classes      = CLASSES,
    shuffle      = True,
    seed         = SEED
)

val_gen = val_test_datagen.flow_from_directory(
    os.path.join(BASE_DIR, 'val'),
    target_size  = IMG_SIZE,
    batch_size   = BATCH_SIZE,
    class_mode   = 'categorical',
    classes      = CLASSES,
    shuffle      = False
)

test_gen = val_test_datagen.flow_from_directory(
    os.path.join(BASE_DIR, 'test'),
    target_size  = IMG_SIZE,
    batch_size   = BATCH_SIZE,
    class_mode   = 'categorical',
    classes      = CLASSES,
    shuffle      = False
)

print(f'\n✅ Train samples : {train_gen.samples}')
print(f'✅ Val samples   : {val_gen.samples}')
print(f'✅ Test samples  : {test_gen.samples}')

## 🏗️ 4. Model Builder — Transfer Learning

In [ ]:
def build_model(base_model_name, num_classes, input_shape=(224, 224, 3)):
    """
    Build a transfer learning model using the specified backbone.
    Architecture: Pretrained Base → GlobalAvgPool → Dense(256) → Dropout → Output
    """
    # ── LOAD BASE MODEL (ImageNet weights, no top) ──────────────
    if base_model_name == 'ResNet50':
        base = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    elif base_model_name == 'DenseNet121':
        base = DenseNet121(weights='imagenet', include_top=False, input_shape=input_shape)
    elif base_model_name == 'MobileNetV3':
        base = MobileNetV3Small(weights='imagenet', include_top=False, input_shape=input_shape)
    else:
        raise ValueError(f'Unknown model: {base_model_name}')
    
    # ── PHASE 1: Freeze base (feature extraction) ───────────────
    base.trainable = False
    
    # ── CUSTOM HEAD ─────────────────────────────────────────────
    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=base.input, outputs=outputs, name=base_model_name)
    return model, base

print('✅ Model builder ready')

## 📏 5. mAP Calculation Utility

In [ ]:
def compute_map(model, generator, num_classes):
    """
    Compute mean Average Precision (mAP) across all classes.
    Uses one-vs-rest strategy with sklearn's average_precision_score.
    """
    generator.reset()
    y_true, y_pred = [], []
    
    steps = len(generator)
    for i, (X_batch, y_batch) in enumerate(generator):
        preds = model.predict(X_batch, verbose=0)
        y_true.append(y_batch)
        y_pred.append(preds)
        if i >= steps - 1:
            break
    
    y_true = np.vstack(y_true)
    y_pred = np.vstack(y_pred)
    
    aps = []
    for c in range(num_classes):
        ap = average_precision_score(y_true[:, c], y_pred[:, c])
        aps.append(ap)
    
    return np.mean(aps), aps

print('✅ mAP utility ready')

## 🚀 6. Training Pipeline

In [ ]:
def train_model(model_name, train_gen, val_gen, num_classes, epochs=50):
    """
    Full training pipeline:
    Phase 1 (25 epochs) — Feature extraction (frozen base)
    Phase 2 (25 epochs) — Fine-tuning (top layers unfrozen)
    """
    print(f'\n{"="*60}')
    print(f'🔧 Training: {model_name}')
    print(f'{"="*60}')
    
    model, base = build_model(model_name, num_classes)
    
    # ── CALLBACKS ───────────────────────────────────────────────
    callbacks = [
        EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7, verbose=1),
        ModelCheckpoint(
            filepath=f'{RESULTS_DIR}/{model_name}_best.keras',
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        )
    ]
    
    # ── PHASE 1: Feature Extraction (frozen base) ────────────────
    phase1_epochs = epochs // 2
    print(f'\n📌 Phase 1: Feature Extraction ({phase1_epochs} epochs, LR={LR_PHASE1})')
    model.compile(
        optimizer=Adam(learning_rate=LR_PHASE1),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    start_time = time.time()
    h1 = model.fit(
        train_gen,
        epochs           = phase1_epochs,
        validation_data  = val_gen,
        callbacks        = callbacks,
        verbose          = 1
    )
    
    # ── PHASE 2: Fine-Tuning (unfreeze top layers) ───────────────
    phase2_epochs = epochs - phase1_epochs
    print(f'\n🔓 Phase 2: Fine-Tuning ({phase2_epochs} epochs, LR={LR_PHASE2})')
    
    # Unfreeze the top 30 layers of the base model
    base.trainable = True
    for layer in base.layers[:-30]:
        layer.trainable = False
    
    model.compile(
        optimizer=Adam(learning_rate=LR_PHASE2),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    h2 = model.fit(
        train_gen,
        initial_epoch    = phase1_epochs,
        epochs           = epochs,
        validation_data  = val_gen,
        callbacks        = callbacks,
        verbose          = 1
    )
    
    total_time = time.time() - start_time
    
    # ── MERGE HISTORIES ─────────────────────────────────────────
    history = {
        'accuracy'    : h1.history['accuracy']     + h2.history['accuracy'],
        'val_accuracy': h1.history['val_accuracy'] + h2.history['val_accuracy'],
        'loss'        : h1.history['loss']         + h2.history['loss'],
        'val_loss'    : h1.history['val_loss']     + h2.history['val_loss'],
    }
    
    # ── SAVE HISTORY ────────────────────────────────────────────
    hist_path = f'{RESULTS_DIR}/{model_name.lower()}_history.json'
    with open(hist_path, 'w') as f:
        json.dump({'history': history, 'training_time_seconds': total_time}, f)
    
    print(f'\n✅ {model_name} done! Training time: {total_time/60:.1f} minutes')
    print(f'   Best val accuracy: {max(history["val_accuracy"]):.4f}')
    
    return model, history, total_time

print('✅ Training pipeline ready')

## 🏋️ 7. Train All 3 Models

In [ ]:
MODEL_NAMES = ['ResNet50', 'DenseNet121', 'MobileNetV3']
trained_models = {}
all_histories  = {}
training_times = {}

for model_name in MODEL_NAMES:
    model, history, t = train_model(model_name, train_gen, val_gen, NUM_CLASSES, EPOCHS)
    trained_models[model_name] = model
    all_histories[model_name]  = history
    training_times[model_name] = t

print('\n🎉 All models trained!')

## 📊 8. Evaluate Models — Accuracy & mAP

In [ ]:
model_params = {
    'ResNet50'    : 25_636_712,
    'DenseNet121' : 8_062_504,
    'MobileNetV3' : 5_483_032
}

print('\n📊 Evaluating models on TEST set...')
print(f'{"Model":<14} {"Test Acc":>10} {"mAP":>10} {"Time (min)":>12} {"Params (M)":>12}')
print('-' * 62)

eval_results = {}

for model_name, model in trained_models.items():
    test_gen.reset()
    loss, acc = model.evaluate(test_gen, verbose=0)
    
    test_gen.reset()
    mean_ap, per_class_ap = compute_map(model, test_gen, NUM_CLASSES)
    
    t_min   = training_times[model_name] / 60
    params  = model_params[model_name] / 1e6
    
    eval_results[model_name] = {
        'test_accuracy' : acc,
        'mAP'           : mean_ap,
        'per_class_ap'  : per_class_ap,
        'training_time' : t_min,
        'params_M'      : params
    }
    
    print(f'{model_name:<14} {acc:>10.4f} {mean_ap:>10.4f} {t_min:>12.1f} {params:>12.1f}')

# Save results
with open(f'{RESULTS_DIR}/eval_results.json', 'w') as f:
    json.dump(eval_results, f, indent=2)

print('\n💾 Results saved to results/eval_results.json')

## 📈 9. Plot Training Curves

In [ ]:
fig, axes = plt.subplots(len(MODEL_NAMES), 2, figsize=(16, 5 * len(MODEL_NAMES)))
fig.suptitle('Training Curves — All Models', fontsize=16, fontweight='bold')

colors = {'train': '#2ECC71', 'val': '#E74C3C'}

for row, model_name in enumerate(MODEL_NAMES):
    hist = all_histories[model_name]
    epochs_range = range(1, len(hist['accuracy']) + 1)
    
    # Accuracy
    ax1 = axes[row][0]
    ax1.plot(epochs_range, hist['accuracy'],     label='Train', color=colors['train'], linewidth=2)
    ax1.plot(epochs_range, hist['val_accuracy'], label='Val',   color=colors['val'],   linewidth=2, linestyle='--')
    ax1.axvline(x=25, color='gray', linestyle=':', alpha=0.7, label='Fine-tune start')
    ax1.set_title(f'{model_name} — Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(alpha=0.3)
    ax1.set_ylim(0, 1)
    
    # Loss
    ax2 = axes[row][1]
    ax2.plot(epochs_range, hist['loss'],     label='Train', color=colors['train'], linewidth=2)
    ax2.plot(epochs_range, hist['val_loss'], label='Val',   color=colors['val'],   linewidth=2, linestyle='--')
    ax2.axvline(x=25, color='gray', linestyle=':', alpha=0.7, label='Fine-tune start')
    ax2.set_title(f'{model_name} — Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: results/training_curves.png')

---
## ✅ Data Science Complete!

**Summary:**
- ✅ ResNet50 trained with transfer learning (50 epochs)
- ✅ DenseNet121 trained with transfer learning (50 epochs)
- ✅ MobileNetV3 trained with transfer learning (50 epochs)
- ✅ Accuracy & mAP recorded for all models
- ✅ Training times recorded
- ✅ Hyperparameter tuning: LR scheduling, Dropout, L2 reg, fine-tuning

**Next step:** Run `03_data_analyst.ipynb` for full visualization & conclusion.